# Pre-requisite
Prior to running this notebook, please ensure you have added the 'evaluation_lakehouse' as a Data Item.

In [ ]:
%pip install -U fabric-data-agent-sdk

# ✅ TASK OVERVIEW

**🎯 Goal**  
Configure your **Data Agent Artifact** using AI instructions, data source setup, and few-shot examples — aiming for **100% correctness** on all questions for the Dataset of your choice.

---

## 🧪 STEP 1: Use the Evaluation Notebook

1. **Update the `YOUR_DATA_AGENT_ARTIFACT_NAME`**  
   - Replace the placeholder with the name of your Data Agent Artifact from in Step 1 of the "Lab 3 Curate Your Data Agent" Document. 
2. **Update the `YOUR_ACCOUNT_NUMBER` Variable**
   - Replace the placeholder with your name

In [ ]:
from fabric.dataagent.client import FabricDataAgentManagement
from fabric.dataagent.evaluation import evaluate_data_agent, get_evaluation_summary, get_evaluation_details, get_evaluation_summary_per_question
#from fabric.dataagent.evaluation.evaluator_api import add_ground_truth_batch
import openai
import os
import re

YOUR_DATA_AGENT_ARTIFACT_NAME = "ECommerceAgent"

YOUR_ACCOUNT_NUMBER = "001"

os.environ["OPENAI_API_VERSION"] = "2023-05-15"

data_agent = FabricDataAgentManagement(YOUR_DATA_AGENT_ARTIFACT_NAME)

## 🔄 STEP 2: Understanding the Groundtruth

This step takes the **groundtruths** and displays the **expected answers** and their associated **queries**.

💡 This is the reference we will use to evaluate your Data Agent's performance.

In [ ]:
csv_path = "Files/Files/benchmark.csv" 

groundtruth_df = (spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .csv(csv_path)
    .limit(1000)
)

df_pandas_groundtruth = groundtruth_df.toPandas()
type(df_pandas_groundtruth)

display(df_pandas_groundtruth)

## 📊 STEP 3: Evaluation

This is the **actual evaluation step**, where a new **conversation thread** is started with the Data Agent for **each question**.

- We use an **LLM-critic approach** to assess the agent’s answer against the expected answer.  
  > 🧠 *LLM-critic*: A language model is used to **judge** whether the generated response matches the expected one — even when the phrasing differs.
  
- ✅ **Note**: We **do not compare** the SQL queries. Only the **final answer text** is evaluated.

---

### 📄 Evaluation Output Columns

| Column           | Description                                                                 |
|------------------|-----------------------------------------------------------------------------|
| **T**            | ✅ Number of correct answers (matches expected output)                      |
| **F**            | ❌ Number of incorrect answers                                               |
| **?**            | 🤔 Unclear — the LLM critic couldn’t decide                                 |
| **%**            | ✔️ Percentage of T out of total (T + F + ?)                                 |
| **failed thread**| 🔗 Links to failed threads (for manual inspection)                          |
| **question**     | 📝 The question from the ground truth                                       |

> 🔁 You can use `num_query_repeat` to ask the same question multiple times to assess variability in responses.

---

### ✅ Next Steps

- Publish your DataAgent
- Inspect the threads that **have mismatches**
- Update:
  - AI instructions
  - Data source instructions
  - Few-shot examples  
- **Re-run the evaluation**


In [ ]:

# The table used to store evaluation results
evaluation_table_name = f'evaluation_output_{YOUR_ACCOUNT_NUMBER}'

# run evaluation
eval_id = evaluate_data_agent(
    df = df_pandas_groundtruth,
    data_agent_name = YOUR_DATA_AGENT_ARTIFACT_NAME,
    data_agent_stage = "production",
    table_name = evaluation_table_name)

df = get_evaluation_summary_per_question(eval_id, verbose=True, table_name = evaluation_table_name)

## 🖨️ STEP 4: Pretty Print the Evaluation Results

🔍 Review the detailed output of your evaluation run — inspect **failures** and **mismatches** to guide your next improvements.

In [ ]:
from fabric.dataagent.evaluation import get_evaluation_details
 
# Table name used during evaluation
table_name = "demo_evaluation_output"
 
# Whether to return all evaluation rows (True) or only failures (False)
get_all_rows = False
 
# Whether to print a summary of the results
verbose = True
 
# Retrieve evaluation details for a specific run
eval_details = get_evaluation_details(
    eval_id,
    evaluation_table_name,
    get_all_rows=get_all_rows,
    verbose=verbose
)

## 🧩 (Optional) STEP 5: Few-Shot Validator

> ⚠️ **Note:** This is for **Lakehouses ONLY**

💡 Get help improving the quality of your few-shot example queries using our **Few-Shot Evaluator**. It leverages LLMs to assess your examples across **3 key areas**:

| Area            | Description                                      |
|-----------------|--------------------------------------------------|
| 🔎 **Clarity**     | Is the natural language question clear?         |
| 🗺️ **Mapping**     | Does the SQL correctly map to the question?     |
| 🔗 **Relatedness** | Is the example relevant to the data source?     |

✅ The evaluator provides a clear breakdown of which few-shots are **effective** and which **need improvement**.

📚 **Learn more:** [Validate Example Queries — Microsoft Docs](https://learn.microsoft.com/en-us/fabric/data-science/data-agent-example-queries#validate-example-queries)

In [ ]:
datasource = data_agent.get_datasources()[0]

# Evaluate few-shot examples using the Data Agent SDK.
# This runs validation on your natural-language/SQL pairs and returns a summary of results.
result = datasource.evaluate_few_shots(batch_size=20)

# Access success and failure cases as pre-computed Pandas DataFrames
success_df = result.success_cases
failure_df = result.failure_cases

print("Success Cases:")
display(success_df)  # Shows examples where the SQL matched the user question

print("Failure Cases:")
display(failure_df)  # Shows examples that need review or improvement